# 04 — Grad-CAM visualization

Generate **Gradient-weighted Class Activation Maps** on the last convolutional block of the image encoder, overlaid on input MRI slices. Useful for qualitative interpretability (which regions drive the class logit).

**Prerequisites:** at least one `*.pt` checkpoint under `experiments/checkpoints/` (default name `best_model.pt`). If `best_model.pt` is missing, the loader uses the **newest** `*.pt` in that folder and prints which file it picked.

Paths are resolved from the **project root** via `utils/paths.py` (not the notebook folder). After pulling code updates, use **Kernel → Restart** and run all cells, or rely on the import cell that clears cached `visualization` / `utils.paths` modules.

Data must exist under `data/raw` as in `configs/config.yaml`.

In [1]:
import sys
sys.path.insert(0, '..')

# Reload local packages if the kernel cached an older copy (avoids stale gradcam_pipeline)
for _k in list(sys.modules):
    if _k == 'utils.paths' or _k == 'visualization' or _k.startswith('visualization.'):
        del sys.modules[_k]

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

%matplotlib inline

from utils.config import load_config
from utils.helpers import set_seed, get_device
from data.dataset_loader import CLASS_NAMES
from visualization.gradcam_pipeline import (
    build_test_loader_for_gradcam,
    export_gradcam_batch,
    gradcam_arrays_for_display,
    load_vl_jepa_from_checkpoint,
)

print('Imports OK (visualization reloaded)')

Imports OK


## Configuration

Adjust `NUM_SAMPLES`, `TARGET_MODE`, and `OUTPUT_DIR` as needed.

In [6]:
cfg = load_config('../configs')
set_seed(cfg['project']['seed'])
device = get_device(cfg['device'])

# None → project-root path + optional fallback to newest *.pt in experiments/checkpoints/
CHECKPOINT_PATH = None  # e.g. 'experiments/checkpoints/best_model.pt' or absolute path

from utils.paths import find_checkpoint_for_inference, project_root

NUM_SAMPLES = 12
TARGET_MODE = 'true_label'   # 'true_label' | 'predicted'
OUTPUT_DIR = Path('../outputs/gradcam_notebook')
OVERLAY_ALPHA = 0.45

print(f'Device: {device}')
print(f'Project root: {project_root()}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')
try:
    _ck = find_checkpoint_for_inference(cfg, CHECKPOINT_PATH)
    print(f'Checkpoint OK: {_ck}')
except FileNotFoundError as e:
    print(e)

Device: mps
Project root: /Users/sreethanubhuvaneshgk/Downloads/desktop/home_folder/alzheimer's_detection/alzheimers_vl_jepa
Output dir: /Users/sreethanubhuvaneshgk/Downloads/desktop/home_folder/alzheimer's_detection/alzheimers_vl_jepa/outputs/gradcam_notebook
Checkpoint OK: /Users/sreethanubhuvaneshgk/Downloads/desktop/home_folder/alzheimer's_detection/alzheimers_vl_jepa/experiments/checkpoints/best_model.pt


## Load model and test data

In [ ]:
model, device = load_vl_jepa_from_checkpoint(cfg, checkpoint_path=CHECKPOINT_PATH, device=device)
test_loader = build_test_loader_for_gradcam(cfg, batch_size=8, num_workers=0)
print(f'Test batches available; first batch will be used for inline plots.')

## Export overlay PNGs

Files are named with the **explained class** (Grad-CAM target) and the **true label** index.

In [ ]:
paths = export_gradcam_batch(
    model,
    test_loader,
    cfg,
    device,
    OUTPUT_DIR,
    num_samples=NUM_SAMPLES,
    target_mode=TARGET_MODE,
    alpha=OVERLAY_ALPHA,
    file_prefix='gradcam',
)
print('Saved:', len(paths), 'files')
for p in paths[:5]:
    print(' ', p)

## Inline figure: original vs. heatmap vs. overlay

Uses the **first** test image in the loader.

In [ ]:
images, _tokens, labels = next(iter(test_loader))
img = images[:1]
true_y = int(labels[0].item())

with torch.no_grad():
    pred = model(img.to(device), _tokens[:1].to(device), labels=labels[:1].to(device))['logits'].argmax(dim=1).item()

explain_class = pred if TARGET_MODE == 'predicted' else true_y
mean = tuple(cfg['dataset'].get('mean', [0.485, 0.456, 0.406]))
std = tuple(cfg['dataset'].get('std', [0.229, 0.224, 0.225]))

rgb, heatmap, overlay = gradcam_arrays_for_display(
    model,
    img,
    target_class=explain_class,
    device=device,
    mean=mean,
    std=std,
    max_seq_len=cfg['text_encoder']['max_seq_len'],
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb)
axes[0].set_title(f'Input (true: {CLASS_NAMES[true_y]})')
axes[0].axis('off')
axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title(f'Grad-CAM (target: {CLASS_NAMES[explain_class]})')
axes[1].axis('off')
axes[2].imshow(overlay)
axes[2].set_title('Overlay')
axes[2].axis('off')
plt.tight_layout()
plt.show()